# Empirical Validation: Do Lindsey's Behavioral Dimensions Match Karin's Data?


This script validates whether the Thai-adapted term dictionaries in
behavioral_features.py actually:
  1. EXIST in Karin's Pantip dataset (hit rate per dictionary)
  2. DISCRIMINATE between sockpuppet vs. non-sockpuppet users (chi-squared,
     effect size, mean difference)
  3. Identify which specific terms are data-driven vs. dead-weight


In [1]:
import re
import json
import argparse
import numpy as np
import pandas as pd
from collections import Counter, defaultdict
from scipy import stats

## SECTION 0: The dictionaries under test (copied from behavioral_features.py)

In [2]:
POLITICAL_TERMS = [
    'ก้าวไกล', 'พท', 'เพื่อไทย', 'ภท', 'ภูมิใจไทย', 'ปชป', 'ประชาธิปัตย์',
    'รทสช', 'รวมไทยสร้างชาติ', 'ชาติไทยพัฒนา', 'ชทพ',
    'พิธา', 'แพทองธาร', 'อุ๊งอิ๊ง', 'เศรษฐา', 'อภิสิทธิ์', 'จุรินทร์',
    'ประยุทธ์', 'ประวิตร', 'พลเอก',
    'เลือกตั้ง', 'นายกรัฐมนตรี', 'นายก', 'ส.ส.', 'สส', 'พรรค',
    'นโยบาย', 'โหวต', 'เขต', 'บัตร', 'หน่วยเลือกตั้ง',
    'รัฐบาล', 'ฝ่ายค้าน', 'รัฐสภา', 'สภา', 'ประชาธิปไตย',
]

ATTACK_TERMS = {
    'โง่': 0.8, 'ไอ้โง่': 0.9, 'ไม่มีสมอง': 0.8, 'ควาย': 0.9,
    'ไอ้ควาย': 1.0, 'โกง': 0.9, 'คอร์รัปชัน': 0.8,
    'สลิ่ม': 0.8, 'แดง': 0.6, 'เหลือง': 0.6, 'เผด็จการ': 0.8,
    'ทหาร': 0.4, 'รัฐประหาร': 0.7, 'ซื้อเสียง': 0.9,
    'ไม่รู้เรื่อง': 0.6, 'พวกนี้': 0.5, 'ไม่เข้าใจ': 0.4,
    'พวกมึง': 0.9, 'มึง': 0.8, 'กู': 0.5,
    'ไอโอ': 0.9, 'io': 0.8, 'บอท': 0.9, 'bot': 0.8,
    'จ้างมา': 0.9, 'รับจ้าง': 0.8, 'ม็อบ': 0.5,
}

EMOTIONAL_TERMS = {
    'ซาบซึ้ง': 0.7, 'ประทับใจ': 0.6, 'ภูมิใจ': 0.7, 'น้ำตาไหล': 0.8,
    'ขนลุก': 0.7, 'อบอุ่น': 0.5, 'ดีใจ': 0.5,
    'น่ากลัว': 0.6, 'หวาดกลัว': 0.7, 'โกรธ': 0.6, 'เกลียด': 0.7,
    'รังเกียจ': 0.7, 'เศร้า': 0.5, 'ผิดหวัง': 0.5,
    'มากๆ': 0.5, 'สุดๆ': 0.6, 'เกินไป': 0.4, 'จริงๆ': 0.3,
    'โคตร': 0.6, 'ห่วย': 0.7, 'แย่มาก': 0.6,
}

LOW_EFFORT_PATTERNS = [
    'เห็นด้วย', 'ถูกต้อง', 'จริง', 'ใช่เลย', 'โอเค', 'ok',
    'ดีมาก', 'เยี่ยม', 'เก่ง', 'สู้ๆ', 'เชียร์',
]

COORDINATION_PHRASES = [
    'มาช่วยกัน', 'ช่วยกันแชร์', 'แชร์ด้วย', 'บอกต่อ',
    'ทุกคนมา', 'ร่วมกัน', '@', 'แท็ก',
]

ALGORITHMIC_TERMS = [
    'แชร์ด้วย', 'ช่วยแชร์', 'กระจายด้วย', 'บอกต่อด้วย',
    'share', 'กด like', 'กดไลค์', 'ไวรัล',
]

AUTHENTICITY_PHRASES = [
    'ปกติไม่ค่อยคอมเมนต์', 'ไม่ค่อยโพสต์', 'เงียบมาตลอด',
    'ครั้งแรกที่', 'ไม่เคยพูดเรื่องการเมือง', 'ในฐานะที่',
    'ขอพูดตรงๆ', 'พูดตามตรง',
]

ALL_DICTIONARIES = {
    'political':      ('list', POLITICAL_TERMS),
    'attack':         ('dict', ATTACK_TERMS),
    'emotional':      ('dict', EMOTIONAL_TERMS),
    'low_effort':     ('list', LOW_EFFORT_PATTERNS),
    'coordination':   ('list', COORDINATION_PHRASES),
    'algorithmic':    ('list', ALGORITHMIC_TERMS),
    'authenticity':   ('list', AUTHENTICITY_PHRASES),
}


## SECTION 1: Load Karin's Dataset

In [3]:
def load_karin_dataset(data_path: str) -> pd.DataFrame:
    """
    Load Karin's Pantip dataset. Supports:
      - JSON (.json / .txt) from Data_Contruct_and_Preprocess output
      - CSV (.csv) if already exported

    Expected columns: user_id, messages, user_sameIP (or IP_check for label)
    """
    if data_path.endswith('.csv'):
        df = pd.read_csv(data_path, encoding='utf-8-sig')
    else:
        with open(data_path, 'r', encoding='utf-8-sig', errors='replace') as f:
            raw = f.read()

        # Try standard JSON first
        try:
            data = json.loads(raw)
        except json.JSONDecodeError as e:
            print(f"[WARNING] Standard JSON failed: {e}")
            print("[WARNING] Attempting repair (strict=False + truncation)...")

            # Strategy 1: truncate at the error position and close the array
            error_pos = e.pos
            # Walk back to the last complete object (find last '}')
            truncated = raw[:error_pos]
            last_brace = truncated.rfind('}')
            if last_brace > 0:
                truncated = truncated[:last_brace + 1]
                # Close the JSON array if needed
                if not truncated.rstrip().endswith(']'):
                    truncated = truncated.rstrip().rstrip(',') + '\n]'
                try:
                    data = json.loads(truncated)
                    print(f"[WARNING] Loaded {len(data)} records (truncated at corrupt entry)")
                except json.JSONDecodeError:
                    # Strategy 2: line-by-line parsing with eval fallback
                    print("[WARNING] Truncation failed. Trying line-by-line eval...")
                    import ast
                    # The file might be a JSON array; try parsing each object
                    data = []
                    # Strip outer brackets and split by },{ pattern
                    content = raw.strip()
                    if content.startswith('['):
                        content = content[1:]
                    if content.endswith(']'):
                        content = content[:-1]
                    # This is a last resort
                    raise RuntimeError(
                        "Could not parse the JSON file. Please re-export from "
                        "Data_Contruct_and_Preprocess.ipynb as CSV instead:\n"
                        "  main_df.to_csv('karin_dataset.csv', index=False)"
                    )
            else:
                raise

        df = pd.DataFrame(data)

    print(f"Loaded {len(df)} posts from {data_path}")
    print(f"Columns: {list(df.columns)}")
    print(f"Unique users (before filter): {df['user_id'].nunique()}")

    # Use the cleaned message column
    text_col = None
    for candidate in ['messages', 'msgClean_', 'msgClean', 'text']:
        if candidate in df.columns:
            text_col = candidate
            break

    if text_col and text_col != 'messages':
        df['messages'] = df[text_col]

    sock_count = (df['is_sockpuppet'] == 1).sum()
    non_sock_count = (df['is_sockpuppet'] == 0).sum()
    print(f"Label distribution: sockpuppet={sock_count}, non-sockpuppet={non_sock_count}")

    return df


## SECTION 2: Term-Level Hit Analysis

In [4]:
def check_term_hits(df: pd.DataFrame):
    """
    For each dictionary, count how many posts contain each term.
    Report: term, total_hits, hit_rate, sockpuppet_hits, non_sock_hits
    """
    results = []
    total_posts = len(df)
    has_labels = (df['is_sockpuppet'] != -1).all()

    for dim_name, (dtype, terms) in ALL_DICTIONARIES.items():
        term_list = list(terms.keys()) if dtype == 'dict' else terms

        for term in term_list:
            # Count posts containing this term
            mask = df['messages'].fillna('').str.contains(
                re.escape(term), case=False, na=False
            )
            total_hits = mask.sum()
            hit_rate = total_hits / total_posts if total_posts > 0 else 0

            row = {
                'dimension': dim_name,
                'term': term,
                'total_hits': total_hits,
                'hit_rate': hit_rate,
                'hit_pct': f"{hit_rate*100:.2f}%",
            }

            if has_labels:
                sock_mask = mask & (df['is_sockpuppet'] == 1)
                non_sock_mask = mask & (df['is_sockpuppet'] == 0)
                n_sock = (df['is_sockpuppet'] == 1).sum()
                n_non_sock = (df['is_sockpuppet'] == 0).sum()

                row['sock_hits'] = sock_mask.sum()
                row['non_sock_hits'] = non_sock_mask.sum()
                row['sock_rate'] = sock_mask.sum() / n_sock if n_sock > 0 else 0
                row['non_sock_rate'] = non_sock_mask.sum() / n_non_sock if n_non_sock > 0 else 0

                # Chi-squared test (term present vs label)
                if total_hits > 0 and total_hits < total_posts:
                    contingency = np.array([
                        [sock_mask.sum(), (df['is_sockpuppet'] == 1).sum() - sock_mask.sum()],
                        [non_sock_mask.sum(), (df['is_sockpuppet'] == 0).sum() - non_sock_mask.sum()]
                    ])
                    try:
                        chi2, p_val, _, _ = stats.chi2_contingency(contingency)
                        row['chi2'] = chi2
                        row['p_value'] = p_val
                        row['significant'] = p_val < 0.05
                    except:
                        row['chi2'] = 0
                        row['p_value'] = 1.0
                        row['significant'] = False
                else:
                    row['chi2'] = 0
                    row['p_value'] = 1.0
                    row['significant'] = False

            results.append(row)

    return pd.DataFrame(results)


## SECTION 3: Dimension-Level Discrimination Analysis

In [5]:
def _analyze_comment_pantip(text: str) -> dict:
    """
    Score a single Pantip comment across 12 behavioral dimensions.
    Self-contained — uses the PANTIP dictionaries defined in SECTION 0.
    All scores in [0, 1].
    """
    null_scores = {
        'emoji_score': 0.0, 'political_score': 0.0, 'attack_score': 0.0,
        'emotional_score': 0.0, 'low_effort_score': 0.0,
        'coordination_score': 0.0, 'algorithmic_score': 0.0,
        'authenticity_score': 0.0, 'length_score': 0.0,
        'repetition_score': 0.0, 'punctuation_score': 0.0, 'caps_score': 0.0,
    }
    if not text or not isinstance(text, str):
        return null_scores

    text = text.strip()
    lower_text = text.lower()

    emoji_pat = re.compile(
        r'[\U0001F600-\U0001F64F]|[\U0001F300-\U0001F5FF]'
        r'|[\U0001F680-\U0001F6FF]|[\U0001F1E0-\U0001F1FF]'
        r'|[\U00002600-\U000026FF]|[\U00002700-\U000027BF]'
        r'|❤️|💚|✌️|🥰|😭|😢|😊|😍|🔥|👏|🙏|💪|😘|🤗|🙌'
    )
    emojis = emoji_pat.findall(text)
    emoji_count = len(emojis)

    text_no_emoji = emoji_pat.sub('', text)
    words = text_no_emoji.split()  # simple whitespace split (fallback)
    word_count = len(words)

    scores = {}

    # 1. Emoji
    if emoji_count == 0:      scores['emoji_score'] = 0.0
    elif emoji_count <= 2:    scores['emoji_score'] = 0.2
    elif emoji_count <= 4:    scores['emoji_score'] = 0.4
    elif emoji_count <= 6:    scores['emoji_score'] = 0.6
    elif emoji_count <= 9:    scores['emoji_score'] = 0.8
    else:                     scores['emoji_score'] = 1.0
    if word_count > 0:
        ratio = emoji_count / word_count
        if ratio > 2:   scores['emoji_score'] = min(1.0, scores['emoji_score'] + 0.3)
        elif ratio > 1: scores['emoji_score'] = min(1.0, scores['emoji_score'] + 0.2)

    # 2. Political
    political_count = sum(1 for t in POLITICAL_TERMS if t in lower_text)
    scores['political_score'] = min(1.0, political_count / max(1, word_count) * 5)

    # 3. Attack
    max_attack = 0.0
    for t, w in ATTACK_TERMS.items():
        if t in lower_text: max_attack = max(max_attack, w)
    scores['attack_score'] = max_attack

    # 4. Emotional
    max_emo = 0.0
    for t, w in EMOTIONAL_TERMS.items():
        if t in lower_text: max_emo = max(max_emo, w)
    scores['emotional_score'] = max_emo

    # 5. Low effort
    low_effort = 0.0
    if word_count <= 3:
        for p in LOW_EFFORT_PATTERNS:
            if p in lower_text: low_effort = 0.8; break
    scores['low_effort_score'] = low_effort

    # 6. Coordination
    coord = 0.0
    for p in COORDINATION_PHRASES:
        if p in lower_text:
            coord = 1.0 if '@' in p else max(coord, 0.4)
    scores['coordination_score'] = coord

    # 7. Algorithmic
    algo = 0.0
    for t in ALGORITHMIC_TERMS:
        if t in lower_text:
            algo = 1.0 if t in ['ไวรัล'] else max(algo, 0.7)
    scores['algorithmic_score'] = algo

    # 8. Authenticity
    auth = 0.0
    for p in AUTHENTICITY_PHRASES:
        if p in lower_text: auth = 0.8; break
    scores['authenticity_score'] = auth

    # 9. Length
    if word_count == 0:      scores['length_score'] = 0.6
    elif word_count <= 2:    scores['length_score'] = 0.4
    elif word_count >= 50:   scores['length_score'] = 0.3
    else:                    scores['length_score'] = 0.0

    # 10. Repetition
    rep = 0.0
    if re.search(r'[!]{3,}', text) or re.search(r'[?]{3,}', text): rep = 0.4
    if re.search(r'(.)\1{3,}', text.lower()): rep = max(rep, 0.3)
    scores['repetition_score'] = rep

    # 11. Punctuation
    punct_count = len(re.findall(r'[!?.,;:]', text))
    if word_count > 0:
        pr = punct_count / word_count
        scores['punctuation_score'] = 0.6 if pr > 0.5 else (0.3 if pr > 0.3 else 0.0)
    else:
        scores['punctuation_score'] = 0.0

    # 12. Caps
    caps_count = sum(1 for c in text if c.isupper())
    if len(text) > 0:
        cr = caps_count / len(text)
        if cr > 0.7 and len(text) > 10: scores['caps_score'] = 0.8
        elif cr > 0.5:                  scores['caps_score'] = 0.4
        else:                           scores['caps_score'] = 0.0
    else:
        scores['caps_score'] = 0.0

    return scores


def check_dimension_discrimination(df: pd.DataFrame):
    """
    For each of the 12 dimensions, check whether the dimension score
    differs between sockpuppet and non-sockpuppet posts.

    Uses Mann-Whitney U test (non-parametric) and reports effect size.
    """
    has_labels = (df['is_sockpuppet'] != -1).all()
    if not has_labels:
        print("[SKIP] No labels available for discrimination analysis.")
        return None

    # Score each post across all 12 dimensions
    print("Scoring all posts across 12 behavioral dimensions...")

    scores_list = []
    for idx, row in df.iterrows():
        text = row.get('messages', '')
        scores = _analyze_comment_pantip(text)
        scores['is_sockpuppet'] = row['is_sockpuppet']
        scores_list.append(scores)

    scores_df = pd.DataFrame(scores_list)

    sock = scores_df[scores_df['is_sockpuppet'] == 1]
    non_sock = scores_df[scores_df['is_sockpuppet'] == 0]

    results = []
    score_cols = [c for c in scores_df.columns if c.endswith('_score')]

    for col in score_cols:
        sock_vals = sock[col].values
        non_sock_vals = non_sock[col].values

        # Basic stats
        sock_mean = np.mean(sock_vals)
        non_sock_mean = np.mean(non_sock_vals)
        sock_nonzero = (sock_vals > 0).mean()
        non_sock_nonzero = (non_sock_vals > 0).mean()

        # Mann-Whitney U test
        try:
            u_stat, p_val = stats.mannwhitneyu(
                sock_vals, non_sock_vals, alternative='two-sided'
            )
            # Effect size: rank-biserial correlation
            n1, n2 = len(sock_vals), len(non_sock_vals)
            effect_size = 1 - (2 * u_stat) / (n1 * n2)
        except:
            p_val = 1.0
            effect_size = 0.0

        results.append({
            'dimension': col,
            'sock_mean': sock_mean,
            'non_sock_mean': non_sock_mean,
            'mean_diff': sock_mean - non_sock_mean,
            'sock_nonzero_pct': f"{sock_nonzero*100:.1f}%",
            'non_sock_nonzero_pct': f"{non_sock_nonzero*100:.1f}%",
            'mann_whitney_p': p_val,
            'effect_size': effect_size,
            'significant': p_val < 0.05,
            'discriminates': p_val < 0.05 and abs(effect_size) > 0.1,
        })

    return pd.DataFrame(results), scores_df

## SECTION 4: Data-Driven Term Discovery (what SHOULD be in the dictionary)

In [6]:
def discover_discriminative_terms(df: pd.DataFrame, top_n: int = 50):
    """
    Instead of using pre-defined dictionaries, discover which terms
    are statistically overrepresented in sockpuppet vs non-sockpuppet posts.

    Uses character n-gram TF-IDF + chi-squared feature selection.
    This is what the professor wants: let the DATA decide the terms.
    """
    has_labels = (df['is_sockpuppet'] != -1).all()
    if not has_labels:
        print("[SKIP] No labels available for term discovery.")
        return None

    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.feature_selection import chi2

    texts = df['messages'].fillna('').tolist()
    labels = df['is_sockpuppet'].values

    # Use word-level tokenization for interpretable results
    # For Thai, character n-grams (2-4) capture subword patterns
    print("Building TF-IDF matrix with character n-grams (2-6)...")
    vectorizer = TfidfVectorizer(
        analyzer='char_wb',
        ngram_range=(2, 6),
        min_df=5,           # term must appear in at least 5 posts
        max_df=0.5,         # ignore terms in >50% of posts
        max_features=10000,
    )

    X = vectorizer.fit_transform(texts)
    feature_names = vectorizer.get_feature_names_out()

    # Chi-squared test for each n-gram against the label
    print(f"Running chi-squared on {len(feature_names)} n-grams...")
    chi2_scores, p_values = chi2(X, labels)

    # Build results table
    ngram_results = pd.DataFrame({
        'ngram': feature_names,
        'chi2': chi2_scores,
        'p_value': p_values,
    }).sort_values('chi2', ascending=False)

    # For each top n-gram, check which class it's associated with
    top_ngrams = ngram_results.head(top_n).copy()

    sock_texts = df[df['is_sockpuppet'] == 1]['messages'].fillna('')
    non_sock_texts = df[df['is_sockpuppet'] == 0]['messages'].fillna('')

    sock_rates = []
    non_sock_rates = []
    for ngram in top_ngrams['ngram']:
        escaped = re.escape(ngram.strip())
        sock_rate = sock_texts.str.contains(escaped, na=False).mean()
        non_sock_rate = non_sock_texts.str.contains(escaped, na=False).mean()
        sock_rates.append(sock_rate)
        non_sock_rates.append(non_sock_rate)

    top_ngrams['sock_rate'] = sock_rates
    top_ngrams['non_sock_rate'] = non_sock_rates
    top_ngrams['direction'] = top_ngrams.apply(
        lambda r: 'SOCK↑' if r['sock_rate'] > r['non_sock_rate'] else 'NON-SOCK↑',
        axis=1
    )

    return top_ngrams


## SECTION 5: Summary Report

In [7]:
def print_summary(term_hits_df, dim_results_df=None, discovered_df=None):
    """Print a human-readable summary of all validation results."""

    print("\n" + "=" * 80)
    print("VALIDATION REPORT: Thai Term Dictionaries vs Karin's Pantip Dataset")
    print("=" * 80)

    # --- Part 1: Hit rates by dimension ---
    print("\n── PART 1: Term Hit Rates by Dimension ──")
    for dim in term_hits_df['dimension'].unique():
        dim_df = term_hits_df[term_hits_df['dimension'] == dim]
        total_terms = len(dim_df)
        terms_with_hits = (dim_df['total_hits'] > 0).sum()
        total_hits = dim_df['total_hits'].sum()

        print(f"\n  {dim.upper()} ({terms_with_hits}/{total_terms} terms found, {total_hits} total hits)")

        # Show each term
        for _, row in dim_df.sort_values('total_hits', ascending=False).iterrows():
            hit_str = f"{row['total_hits']:>5} hits ({row['hit_pct']})"
            if 'significant' in row and row['total_hits'] > 0:
                sig_str = " ★" if row.get('significant', False) else "  "
                sock_str = f"sock={row.get('sock_hits', '?'):>4}, non={row.get('non_sock_hits', '?'):>4}"
                print(f"    {row['term']:<20} {hit_str}  {sock_str} {sig_str}")
            else:
                marker = "✓" if row['total_hits'] > 0 else "✗"
                print(f"    {marker} {row['term']:<20} {hit_str}")

    # --- Dead terms ---
    dead_terms = term_hits_df[term_hits_df['total_hits'] == 0]
    if len(dead_terms) > 0:
        print(f"\n── DEAD TERMS (0 hits in dataset): {len(dead_terms)} terms ──")
        for _, row in dead_terms.iterrows():
            print(f"    ✗ [{row['dimension']}] {row['term']}")

    # --- Significant terms ---
    if 'significant' in term_hits_df.columns:
        sig_terms = term_hits_df[(term_hits_df['significant'] == True) & (term_hits_df['total_hits'] > 0)]
        print(f"\n── STATISTICALLY SIGNIFICANT TERMS (p < 0.05): {len(sig_terms)} terms ──")
        for _, row in sig_terms.sort_values('chi2', ascending=False).iterrows():
            direction = "SOCK↑" if row.get('sock_rate', 0) > row.get('non_sock_rate', 0) else "NON-SOCK↑"
            print(f"    ★ [{row['dimension']}] {row['term']:<20} χ²={row['chi2']:.2f}, p={row['p_value']:.4f}, {direction}")

    # --- Part 2: Dimension-level discrimination ---
    if dim_results_df is not None:
        print(f"\n── PART 2: Dimension-Level Discrimination (Mann-Whitney U) ──")
        print(f"{'Dimension':<25} {'Sock Mean':>10} {'Non-Sock':>10} {'p-value':>10} {'Effect':>8} {'Disc?':>6}")
        print("-" * 75)
        for _, row in dim_results_df.sort_values('mann_whitney_p').iterrows():
            disc = "YES" if row['discriminates'] else "no"
            print(f"{row['dimension']:<25} {row['sock_mean']:>10.4f} {row['non_sock_mean']:>10.4f} "
                  f"{row['mann_whitney_p']:>10.4f} {row['effect_size']:>8.3f} {disc:>6}")

    # --- Part 3: Data-driven discovery ---
    if discovered_df is not None:
        print(f"\n── PART 3: Top Data-Driven Discriminative N-grams ──")
        print("  (These are what the DATA says matters, not what we hardcoded)")
        print(f"{'N-gram':<20} {'χ²':>10} {'Sock Rate':>10} {'Non-Sock':>10} {'Direction':>12}")
        print("-" * 65)
        for _, row in discovered_df.head(30).iterrows():
            print(f"{row['ngram']:<20} {row['chi2']:>10.2f} {row['sock_rate']:>10.3f} "
                  f"{row['non_sock_rate']:>10.3f} {row['direction']:>12}")

    # --- Final verdict ---
    print("\n" + "=" * 80)
    print("VERDICT")
    print("=" * 80)

    total_terms_count = len(term_hits_df)
    found_count = (term_hits_df['total_hits'] > 0).sum()
    found_pct = found_count / total_terms_count * 100

    print(f"  Terms in dictionaries: {total_terms_count}")
    print(f"  Terms found in data:   {found_count} ({found_pct:.1f}%)")
    print(f"  Terms with 0 hits:     {total_terms_count - found_count}")

    if 'significant' in term_hits_df.columns:
        sig_count = term_hits_df[
            (term_hits_df['significant'] == True) & (term_hits_df['total_hits'] > 0)
        ].shape[0]
        print(f"  Statistically significant (p<0.05): {sig_count}")
        print(f"  → {sig_count}/{found_count} found terms actually discriminate")

    if dim_results_df is not None:
        disc_dims = dim_results_df[dim_results_df['discriminates'] == True]
        print(f"\n  Dimensions that discriminate: {len(disc_dims)}/12")
        if len(disc_dims) > 0:
            for _, r in disc_dims.iterrows():
                print(f"    ✓ {r['dimension']}")

        non_disc = dim_results_df[dim_results_df['discriminates'] == False]
        if len(non_disc) > 0:
            print(f"  Dimensions that DON'T discriminate: {len(non_disc)}/12")
            for _, r in non_disc.iterrows():
                print(f"    ✗ {r['dimension']}")

## Main

In [8]:
DATA_PATH = '00_Datasets_After_preprocessed.csv'   # ← your Karin dataset path
SKIP_SCORING   = False   # set True to skip Part 2 (dimension scoring)
SKIP_DISCOVERY = False   # set True to skip Part 3 (data-driven term discovery)

df = load_karin_dataset(DATA_PATH)

# Part 1: Term hit analysis
print("\n" + "=" * 60)
print("PART 1: Checking term hits in dataset...")
print("=" * 60)
term_hits_df = check_term_hits(df)
term_hits_df.to_csv('term_hits_validation.csv', index=False)
print("Saved: term_hits_validation.csv")

# Part 2: Dimension discrimination
dim_results_df = None
scores_df = None
if not SKIP_SCORING:
    print("\n" + "=" * 60)
    print("PART 2: Scoring dimensions and testing discrimination...")
    print("=" * 60)
    dim_results_df, scores_df = check_dimension_discrimination(df)
    if dim_results_df is not None:
        dim_results_df.to_csv('dimension_discrimination.csv', index=False)
        print("Saved: dimension_discrimination.csv")

# Part 3: Data-driven discovery
discovered_df = None
if not SKIP_DISCOVERY:
    print("\n" + "=" * 60)
    print("PART 3: Discovering discriminative terms from data...")
    print("=" * 60)
    discovered_df = discover_discriminative_terms(df)
    if discovered_df is not None:
        discovered_df.to_csv('discovered_terms.csv', index=False)
        print("Saved: discovered_terms.csv")

# Print summary
print_summary(term_hits_df, dim_results_df, discovered_df)


Loaded 5457 posts from 00_Datasets_After_preprocessed.csv
Columns: ['thread_id', 'topic', 'num_0', 'num_1', 'user_id', 'messages', 'length', 'tag', 'datetime', 'user_react', 'user_type', 'is_sockpuppet', 'sockpuppet_group_id']
Unique users (before filter): 394
Label distribution: sockpuppet=2390, non-sockpuppet=3067

PART 1: Checking term hits in dataset...
Saved: term_hits_validation.csv

PART 2: Scoring dimensions and testing discrimination...
Scoring all posts across 12 behavioral dimensions...
Saved: dimension_discrimination.csv

PART 3: Discovering discriminative terms from data...
Building TF-IDF matrix with character n-grams (2-6)...
Running chi-squared on 10000 n-grams...
Saved: discovered_terms.csv

VALIDATION REPORT: Thai Term Dictionaries vs Karin's Pantip Dataset

── PART 1: Term Hit Rates by Dimension ──

  POLITICAL (35/36 terms found, 5504 total hits)
    พรรค                   894 hits (16.38%)  sock= 365, non= 529   
    ก้าวไกล                565 hits (10.35%)  sock= 